# Using the symbolic regression model

While in the [quickstart notebook](./quickstart.ipynb) we have talked about ML models stored on Hugging Face, this package also offers a symbolic regression model, i.e. a data-driven analytical model defined by a set of equations.

In [1]:
import mammos_ai
import mammos_entity as me

/home/petrocch/repos/mammos/mammos-ai/.pixi/envs/default/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The model equations

This model was trained on hard magnet-only simulated data for single grain cubic particles with $L = 50$ nm edge length, and external field parallel to the anisotropy axis.

It gives analytical equations for the extrinsic properties $H_{\mathrm{c}}$, $M_{\mathrm{r}}$, $BH_{\mathrm{max}}$ from intrinsic properties $Ms$, $A$, and $K$.

The model works with the following rescaling quantities:

- the anisotropy field $H_{\mathrm{A}}$:

  $$
  H_{\mathrm{A}} := \frac{2K}{\mu_0 M_{\mathrm{s}}}
  $$

- the energy product scaling variable $BH_{\mathrm{s}}$:

  $$
  BH_{\mathrm{s}} := \frac{\mu_0 M_{\mathrm{s}}^2}{4}
  $$

- the hardness parameter $\kappa$:

  $$
  \kappa := \sqrt{\frac{K}{\mu_0 M_{\mathrm{s}}^2}}
  $$

- the exchange length $\ell_{\mathrm{ex}}$:

  $$
  \ell_{\mathrm{ex}} := \sqrt{\frac{2 A}{\mu_0 M_{\mathrm{s}}^2}}
  $$

- the reduced grain size $\tilde{L}$:

  $$
  \tilde{L} := \frac{L}{\ell_{\mathrm{ex}}}
  $$

Then, the model is defined by the following equations:

$$
H_\mathrm{c} &= \left[\alpha - n\,\frac{\ln\tilde{L}}{\kappa}\right] H_\mathrm{A}, \\ 
M_\mathrm{r} &= \left(1 - \varepsilon_m \frac{\tilde{L}}{\kappa^4}\right) M_\mathrm{s}, \\
BH_\mathrm{max} &= \left(1 - \varepsilon_b \frac{\tilde{L}}{\kappa^4}\right)^{\!2} BH_{\mathrm{s}},
$$

with fitted constants $\alpha = 0.942$, $n = 0.0921$, $\varepsilon_m = 5.18 \times 10^{-5}$, and $\varepsilon_b = 7.81 \times 10^{-5}$.

## Use the model via `mammos_ai`

The model has the same interface as the `random_forest` models and only the model string needs to be specified.

In [2]:
Ms = me.Ms(1e6)
A = me.A(1e-12)
K = me.Ku(1e6)

extrinsic = mammos_ai.Hc_Mr_BHmax_from_Ms_A_K(Ms, A, K, model="cube50_singlegrain_symbolic_regression_v1.0")
extrinsic

ExtrinsicProperties(Hc=Entity(ontology_label='CoercivityHcExternal', value=np.float64(894604.3352842493), unit='A / m'), Mr=Entity(ontology_label='Remanence', value=np.float64(996758.0267515216), unit='A / m'), BHmax=Entity(ontology_label='MaximumEnergyProduct', value=np.float64(311095.55410614907), unit='J / m3'))

We can access the individual extrinsic properties as follows:

In [3]:
extrinsic.Hc

Entity(ontology_label='CoercivityHcExternal', value=np.float64(894604.3352842493), unit='A / m')

In [4]:
extrinsic.Mr

Entity(ontology_label='Remanence', value=np.float64(996758.0267515216), unit='A / m')

In [5]:
extrinsic.BHmax

Entity(ontology_label='MaximumEnergyProduct', value=np.float64(311095.55410614907), unit='J / m3')

Differently from the `random_forest` models, this function does not check if the sample is a hard magnet based on the intrinsic properties. Therefore it might give physically wrong values for any of the extrinsic properties if used with a set of parameter inside the training range but still relating to soft magnets.

As with the other models, the metadata can be accessed as follows:

In [8]:
mammos_ai.Hc_Mr_BHmax_from_Ms_A_K_metadata(model="cube50_singlegrain_symbolic_regression_v1.0")

{'model_name': 'cube50_singlegrain_symbolic_regression_v1.0',
 'description': 'Symbolic regression model trained on extended simulated data for single grain cubic particles with 50 nm edge length with the external field applied parallel to the anisotropy axis.',
 'training_data_range': {'Ms': (Entity(ontology_label='SpontaneousMagnetization', value=np.float64(79577.47150262764), unit='A / m'),
   Entity(ontology_label='SpontaneousMagnetization', value=np.float64(3978873.5751313814), unit='A / m')),
  'A': (Entity(ontology_label='ExchangeStiffnessConstant', value=np.float64(1e-13), unit='J / m'),
   Entity(ontology_label='ExchangeStiffnessConstant', value=np.float64(1e-11), unit='J / m')),
  'K': (Entity(ontology_label='UniaxialAnisotropyConstant', value=np.float64(10000.0), unit='J / m3'),
   Entity(ontology_label='UniaxialAnisotropyConstant', value=np.float64(10000000.0), unit='J / m3'))},
 'input_parameters': ['Ms (A/m)', 'A (J/m)', 'K1 (J/m^3)'],
 'output_parameters': ['Hc (A/m)', '

```{note}
The symbolic regression model does not have a hard magnet classifier model nor a function `is_hard_magnet_from_Ms_A_K`.
```